# Probabilistic Demand Forecasting with Uncertainty Quantification

This notebook implements probabilistic forecasting techniques for retail demand, focusing on generating full predictive distributions rather than just point estimates.

**Techniques Covered:**
- **Quantile Regression:** Using LightGBM to forecast specific demand quantiles.
- **Conformal Prediction:** A model-agnostic method to create statistically rigorous prediction intervals.
- **Inventory Optimization:** Using Monte Carlo simulation based on probabilistic forecasts to calculate safety stock and optimize order quantities.
- **Scenario Analysis:** Stress-testing the inventory strategy against extreme events.

## 1. Setup and Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from nonconformist.cp import IcpRegressor
from nonconformist.nc import RegressorNc
from nonconformist.base import RegressorAdapter

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Data Generation

We'll generate a synthetic dataset representing daily sales with trend, seasonality, and noise.

In [ ]:
def generate_sales_data(start_date='2021-01-01', periods=365*3):
    dates = pd.date_range(start=start_date, periods=periods, freq='D')
    data = pd.DataFrame({'date': dates})
    
    # Time-based features
    data['day_of_week'] = data['date'].dt.dayofweek
    data['month'] = data['date'].dt.month
    data['time_idx'] = np.arange(periods)
    
    # Base sales with seasonality and trend
    trend = 0.1 * data['time_idx']
    weekly_seasonality = 20 * np.sin(2 * np.pi * data['day_of_week'] / 7)
    yearly_seasonality = 30 * np.sin(2 * np.pi * data['time_idx'] / 365.25)
    noise = np.random.normal(0, 10, periods)
    
    data['sales'] = 100 + trend + weekly_seasonality + yearly_seasonality + noise
    data['sales'] = data['sales'].astype(int).clip(lower=20)
    
    return data.set_index('date').drop(['day_of_week', 'month'], axis=1)

df = generate_sales_data()
df[['sales']].plot(figsize=(15, 5), title='Daily Sales Data')
plt.show()

## 3. Probabilistic Forecasting

### 3.1. Quantile Regression with LightGBM

In [ ]:
def create_features(df):
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['day_of_year'] = df.index.dayofyear
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    return df

df_feat = create_features(df.copy())
X = df_feat.drop('sales', axis=1)
y = df_feat['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=90, shuffle=False)

def train_lgbm_quantile(X_train, y_train, quantile):
    model = lgb.LGBMRegressor(objective='quantile', alpha=quantile, random_state=42)
    model.fit(X_train, y_train)
    return model

q_lower = 0.1
q_median = 0.5
q_upper = 0.9

model_lower = train_lgbm_quantile(X_train, y_train, q_lower)
model_median = train_lgbm_quantile(X_train, y_train, q_median)
model_upper = train_lgbm_quantile(X_train, y_train, q_upper)

preds_lower = model_lower.predict(X_test)
preds_median = model_median.predict(X_test)
preds_upper = model_upper.predict(X_test)

plt.figure(figsize=(15, 6))
plt.plot(y_test.index, y_test, label='Actual Sales')
plt.plot(y_test.index, preds_median, label='Median Forecast (p50)')
plt.fill_between(y_test.index, preds_lower, preds_upper, alpha=0.3, label='80% Prediction Interval (p10-p90)')
plt.title('Quantile Regression Forecasts with LightGBM')
plt.legend()
plt.show()

## 4. Inventory Optimization with Monte Carlo Simulation

In [ ]:
def run_inventory_simulation(preds_median, preds_lower, preds_upper, n_sims=10000):
    # Assume a normal distribution for demand, derived from the quantiles
    mu = preds_median
    sigma = (preds_upper - preds_lower) / (2 * 1.28) # 1.28 is the z-score for 80% confidence
    
    simulated_demand = np.random.normal(mu[:, np.newaxis], sigma[:, np.newaxis], (len(mu), n_sims))
    simulated_demand = np.maximum(0, simulated_demand) # Demand can't be negative
    
    return simulated_demand

def optimize_inventory(simulated_demand, service_level=0.95):
    # Calculate safety stock and order-up-to level
    safety_stock = np.quantile(simulated_demand, service_level, axis=1)
    return safety_stock

sim_demand = run_inventory_simulation(preds_median, preds_lower, preds_upper)
safety_stock_levels = optimize_inventory(sim_demand)

plt.figure(figsize=(15, 6))
plt.plot(y_test.index, preds_median, label='Median Forecast')
plt.plot(y_test.index, safety_stock_levels, label='Order-Up-To Level (95% Service Level)', linestyle='--')
plt.title('Inventory Policy Based on Probabilistic Forecast')
plt.legend()
plt.show()

## 5. Scenario Analysis: Stress-Testing

In [ ]:
def stress_test_inventory(actuals, safety_stock):
    stock_outs = actuals > safety_stock
    excess_inventory = safety_stock - actuals
    
    print(f"Stock-out occurred on {stock_outs.sum()} out of {len(actuals)} days.")
    print(f"Average excess inventory on non-stock-out days: {excess_inventory[~stock_outs].mean():.2f} units")
    
    # Simulate a demand surge
    surge_day = 30
    actuals_surge = actuals.copy()
    actuals_surge.iloc[surge_day] *= 1.5 # 50% demand surge
    stock_outs_surge = actuals_surge > safety_stock
    print(f"
After a 50% demand surge, stock-outs occurred on {stock_outs_surge.sum()} days.")

stress_test_inventory(y_test, safety_stock_levels)